In [1]:
pip install pandas jiwer datasets transformers evaluate openpyxl

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

df = pd.read_excel("Dataset_Challenge_2.xlsx")  # adjust path if needed
df.head()


,English Source,MT System,Post-Edit Text,Nature of change/Comments
0,app store [TAG] apps,tienda de aplicaciones[TAG],Tienda de aplicaciones[TAG] aplicaciones,Missing content
1,tap 7 times on the background,toca 7 veces en el fondo de la pantalla,toca 7 veces el fondo de la pantalla,Ungrammatical.
2,pair,vincular,sincronizar,As per glossary.
3,Use [TAG] device to…,Usa [TAG] un dispositivo para…,Usa el dispositivo de [TAG] para…,Wrong placement of the tag.
4,This will link [TAG] device with yours.,Esto vinculará [TAG] dispositivo con tuyo.,Esto vinculará el dispositivo de [TAG] con el ...,Wrong placement of the tag and missing article...


In [3]:
import jiwer

def sentence_ter(hyp, ref):
    # Simple HTER approximation using word-level WER
    return jiwer.wer(ref, hyp)

# Rename columns for convenience
df = df.rename(columns={
    "English Source": "src_en",
    "MT System": "mt_es",
    "Post-Edit Text": "pe_es",
    "Nature of Change/Comments": "comments"
})

# Drop rows with missing values just in case
df = df.dropna(subset=["src_en", "mt_es", "pe_es"]).reset_index(drop=True)

ters = []
for _, row in df.iterrows():
    mt = str(row["mt_es"])
    pe = str(row["pe_es"])
    ter = sentence_ter(mt, pe)
    ters.append(ter)

df["hter_label"] = ters

print(df[["src_en", "mt_es", "pe_es", "hter_label"]].head())

# Save QE dataset with HTER labels
df.to_csv("qe_dataset_with_hter.csv", index=False)


                                    src_en  \
0                     app store [TAG] apps   
1            tap 7 times on the background   
2                                     pair   
3                     Use [TAG] device to…   
4  This will link [TAG] device with yours.   

                                        mt_es  \
0                 tienda de aplicaciones[TAG]   
1     toca 7 veces en el fondo de la pantalla   
2                                    vincular   
3              Usa [TAG] un dispositivo para…   
4  Esto vinculará [TAG] dispositivo con tuyo.   

                                               pe_es  hter_label  
0           Tienda de aplicaciones[TAG] aplicaciones    0.500000  
1               toca 7 veces el fondo de la pantalla    0.125000  
2                                        sincronizar    1.000000  
3                  Usa el dispositivo de [TAG] para…    0.666667  
4  Esto vinculará el dispositivo de [TAG] con el ...    0.444444  


In [4]:
from datasets import Dataset

df_qe = pd.read_csv("qe_dataset_with_hter.csv")

# Build model input text: EN + MT_ES
df_qe["input_text"] = df_qe.apply(
    lambda r: f"EN: {r['src_en']}\nES_MT: {r['mt_es']}",
    axis=1
)

df_qe[["input_text", "hter_label"]].head()


,input_text,hter_label
0,EN: app store [TAG] apps\nES_MT: tienda de apl...,0.500000
1,EN: tap 7 times on the background\nES_MT: toca...,0.125000
2,EN: pair\nES_MT: vincular,1.000000
3,EN: Use [TAG] device to…\nES_MT: Usa [TAG] un ...,0.666667
4,EN: This will link [TAG] device with yours.\nE...,0.444444


In [5]:
hf_qe = Dataset.from_pandas(df_qe[["input_text", "hter_label"]])
splits = hf_qe.train_test_split(test_size=0.2, seed=42)

train_ds = splits["train"]
val_ds = splits["test"]

len(train_ds), len(val_ds)


(54, 14)

In [6]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import numpy as np
import torch

MODEL_NAME = "bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    out = tokenizer(
        batch["input_text"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )
    out["labels"] = batch["hter_label"]
    return out

tokenized_train = train_ds.map(tokenize, batched=True, remove_columns=["input_text", "hter_label"])
tokenized_val   = val_ds.map(tokenize, batched=True, remove_columns=["input_text", "hter_label"])

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    problem_type="regression",
)

model


Map:   0%|          | 0/54 [00:00<?, ? examples/s]

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1

In [7]:
import evaluate

metric_pearson = evaluate.load("pearsonr")
metric_spearman = evaluate.load("spearmanr")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.squeeze()  # shape (N,)

    pearson = metric_pearson.compute(predictions=preds, references=labels)["pearsonr"]
    spearman = metric_spearman.compute(predictions=preds, references=labels)["spearmanr"]
    mse = float(np.mean((preds - labels) ** 2))

    return {"pearson": pearson, "spearman": spearman, "mse": mse}


In [8]:
batch_size = 8

training_args = TrainingArguments(
    output_dir="./qe_mbert_regression",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=2,   # you can increase if it’s fast enough
    weight_decay=0.01,
    logging_steps=10,
    save_total_limit=1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


C:\Users\santh\AppData\Local\Temp\ipykernel_9108\878026293.py:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [9]:
trainer.train()


C:\Users\santh\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,0.340600


TrainOutput(global_step=14, training_loss=0.31073943206242155, metrics={'train_runtime': 324.0824, 'train_samples_per_second': 0.333, 'train_steps_per_second': 0.043, 'total_flos': 14207869421568.0, 'train_loss': 0.31073943206242155, 'epoch': 2.0})

In [10]:
metrics = trainer.evaluate()
metrics


C:\Users\santh\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.5193702578544617,
 'eval_pearson': 0.21717725391376186,
 'eval_spearman': 0.44485849960861334,
 'eval_mse': 0.5193701982498169,
 'eval_runtime': 11.022,
 'eval_samples_per_second': 1.27,
 'eval_steps_per_second': 0.181,
 'epoch': 2.0}

In [11]:
def error_type_from_comment(text):
    text = str(text).lower()
    if "terminology" in text:
        return "terminology"
    if "grammar" in text or "fluency" in text:
        return "fluency"
    if "missing" in text or "omission" in text:
        return "omission"
    if "addition" in text:
        return "addition"
    if "mistranslation" in text or "meaning" in text:
        return "mistranslation"
    return "other"

df_qe["error_type"] = df_qe["Nature of Change/Comments"].apply(error_type_from_comment)
df_qe["error_type"].value_counts()



KeyError: 'Nature of Change/Comments'

In [12]:
def error_type_from_comment(text):
    text = str(text).lower()
    if "terminology" in text:
        return "terminology"
    if "grammar" in text or "fluency" in text:
        return "fluency"
    if "missing" in text or "omission" in text:
        return "omission"
    if "addition" in text:
        return "addition"
    if "mistranslation" in text or "meaning" in text:
        return "mistranslation"
    return "other"

df_qe["error_type"] = df_qe["comments"].apply(error_type_from_comment)
df_qe["error_type"].value_counts()


KeyError: 'comments'

In [13]:
# fill na and apply
df_qe["comments"] = df_qe["comments"].fillna("").astype(str)
df_qe["error_type"] = df_qe["comments"].apply(error_type_from_comment)

# 6) Show results
print("\nValue counts for 'error_type':")
print(df_qe["error_type"].value_counts(dropna=False))

print("\nPreview (first 10 rows) of comments -> error_type:")
display(df_qe[["comments", "error_type"]].head(10))

# 7) Save a snapshot so you can inspect the created file
try:
    df_qe.to_csv("qe_dataset_with_comments_checked.csv", index=False)
    print("\nSaved snapshot: qe_dataset_with_comments_checked.csv")
except Exception as e:
    print("\nCould not save snapshot:", e)

# Push df_qe back to globals for later cells
globals()['df_qe'] = df_qe
globals()['df'] = df_qe


KeyError: 'comments'

In [14]:
# SAFE cell: ensure 'comments' exists, then apply error_type mapping (no KeyError)
import pandas as pd, os, re
FNAME = "Dataset_Challenge_2.xlsx"

# 1) Load dataframe fresh (preferred) or use existing variables
if os.path.exists(FNAME):
    df_qe = pd.read_excel(FNAME)
    print(f"Loaded file: {FNAME}")
else:
    if 'df_qe' in globals():
        df_qe = globals()['df_qe']
        print("Using existing variable: df_qe")
    elif 'df' in globals():
        df_qe = globals()['df']
        print("Using existing variable: df")
    else:
        raise FileNotFoundError(f"Could not find {FNAME} and no df variables present. Upload the file or set path.")

# 2) Show exact columns (repr to reveal spaces)
print("\n=== Columns in dataframe (with repr) ===")
for i, c in enumerate(df_qe.columns):
    print(i, repr(c))

# 3) Try to find a comments-like column robustly
cols = list(df_qe.columns)
cols_lc = [str(c).lower() for c in cols]

# Candidate heuristics (ordered)
candidates = []
# exact common names
for name in ["nature of change/comments", "nature of change / comments", "nature of change", 
             "nature of change/comments", "nature_of_change_comments", "comments", "comment", "notes", "remarks"]:
    for orig in cols:
        if str(orig).strip().lower() == name:
            candidates.append(orig)
# substring matches
if not candidates:
    for orig, low in zip(cols, cols_lc):
        if any(x in low for x in ["nature", "change", "comment", "comments", "note", "remark"]):
            candidates.append(orig)
# if still none, fallback to columns containing slash or long names
if not candidates:
    for orig in cols:
        if len(str(orig)) > 20 and "/" in str(orig):
            candidates.append(orig)

# 4) Decide what to do
chosen = None
if candidates:
    chosen = candidates[0]
    print(f"\nAuto-detected comment candidate column: {repr(chosen)}")
    # if chosen already exactly 'comments' leave as is, else try to create/rename
    if chosen != "comments":
        if "comments" in df_qe.columns:
            # don't overwrite existing 'comments'; create comments_auto
            df_qe["comments_auto_from_detected"] = df_qe[chosen].astype(str)
            print("Existing 'comments' column present — created 'comments_auto_from_detected' from detected column.")
        else:
            # rename detected -> comments
            df_qe = df_qe.rename(columns={chosen: "comments"})
            print(f"Renamed {repr(chosen)} -> 'comments'")
else:
    # no candidate found: create empty comments column to avoid KeyError
    print("\nNo comments-like column detected. Creating an empty 'comments' column.")
    df_qe["comments"] = ""

# 5) Final safety: ensure 'comments' now exists
if "comments" not in df_qe.columns:
    print("Unexpected: 'comments' still not present after attempts. Creating empty 'comments'.")
    df_qe["comments"] = ""

print("\nFinal columns (repr):")
for i, c in enumerate(df_qe.columns):
    print(i, repr(c))

# 6) Define mapping and apply safely
def error_type_from_comment(text):
    text = str(text).lower()
    if "terminology" in text:
        return "terminology"
    if "grammar" in text or "fluency" in text:
        return "fluency"
    if "missing" in text or "omission" in text:
        return "omission"
    if "addition" in text:
        return "addition"
    if "mistranslation" in text or "meaning" in text:
        return "mistranslation"
    return "other"

# fill NA and apply
df_qe["comments"] = df_qe["comments"].fillna("").astype(str)
df_qe["error_type"] = df_qe["comments"].apply(error_type_from_comment)

print("\nerror_type value counts:")
print(df_qe["error_type"].value_counts(dropna=False))

print("\nSample rows (comments -> error_type):")
display(df_qe[["comments", "error_type"]].head(15))

# 7) Save snapshot for inspection
outpath = "qe_dataset_comments_fixed.csv"
df_qe.to_csv(outpath, index=False)
print(f"\nSaved snapshot: {outpath}")

# 8) Push back to globals for future cells
globals()['df_qe'] = df_qe
globals()['df'] = df_qe

print("\nDone — 'comments' ensured and mapping applied.")


Loaded file: Dataset_Challenge_2.xlsx

=== Columns in dataframe (with repr) ===
0 'English Source'
1 'MT System'
2 'Post-Edit Text'
3 'Nature of change/Comments'

Auto-detected comment candidate column: 'Nature of change/Comments'
Renamed 'Nature of change/Comments' -> 'comments'

Final columns (repr):
0 'English Source'
1 'MT System'
2 'Post-Edit Text'
3 'comments'

error_type value counts:
error_type
other             37
mistranslation    20
omission           4
fluency            4
terminology        3
Name: count, dtype: int64

Sample rows (comments -> error_type):


,comments,error_type
0,Missing content,omission
1,Ungrammatical.,other
2,As per glossary.,other
3,Wrong placement of the tag.,other
4,Wrong placement of the tag and missing article...,omission
5,Wrong punctuation and missing space.,omission
6,"Mistranslation, meaning has not been understood",mistranslation
7,"Mistranslation, meaning has not been understood",mistranslation
8,"Mistranslation, meaning has not been understood",mistranslation
9,"Mistranslation, contextual",mistranslation



Saved snapshot: qe_dataset_comments_fixed.csv

Done — 'comments' ensured and mapping applied.


In [15]:
pip install -q transformers datasets evaluate pandas openpyxl jiwer


Note: you may need to restart the kernel to use updated packages.


In [18]:
import pandas as pd
import numpy as np
import os
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

print("Working dir files:", os.listdir('.')[:40])

# --- Load df_qe or reload from Excel ---
if 'df_qe' in globals():
    df_qe = df_qe
    print("Using existing df_qe (rows):", len(df_qe))
else:
    df_qe = pd.read_excel("Dataset_Challenge_2.xlsx")
    print("Loaded Dataset_Challenge_2.xlsx (rows):", len(df_qe))

# --- RENAME columns to the required names ---
df_qe = df_qe.rename(columns={
    "English Source": "src_en",
    "MT System": "mt_es",
    "Post-Edit Text": "pe_es"
})

# Verify rename
print("Columns after renaming:", list(df_qe.columns))

# Display first rows
display(df_qe.head())


Working dir files: ['.ipynb_checkpoints', 'AI_ML_Technical_Test_Final_Report.pdf', 'challenge_1.ipynb', 'Challenge_1_Summary_Report.pdf', 'Challenge_2.ipynb', 'Dataset_Challenge_1.xlsx', 'Dataset_Challenge_2.xlsx', 'outputs_mb_ende_nl', 'qe_dataset_comments_fixed.csv', 'qe_dataset_with_hter.csv', 'qe_mbert_regression']
Using existing df_qe (rows): 68
Columns after renaming: ['src_en', 'mt_es', 'pe_es', 'comments', 'error_type', 'input_text']


,src_en,mt_es,pe_es,comments,error_type,input_text
0,app store [TAG] apps,tienda de aplicaciones[TAG],Tienda de aplicaciones[TAG] aplicaciones,Missing content,omission,EN: \nES_MT:
1,tap 7 times on the background,toca 7 veces en el fondo de la pantalla,toca 7 veces el fondo de la pantalla,Ungrammatical.,other,EN: \nES_MT:
2,pair,vincular,sincronizar,As per glossary.,other,EN: \nES_MT:
3,Use [TAG] device to…,Usa [TAG] un dispositivo para…,Usa el dispositivo de [TAG] para…,Wrong placement of the tag.,other,EN: \nES_MT:
4,This will link [TAG] device with yours.,Esto vinculará [TAG] dispositivo con tuyo.,Esto vinculará el dispositivo de [TAG] con el ...,Wrong placement of the tag and missing article...,omission,EN: \nES_MT:


In [19]:
# Build model input combining EN source + MT output
df_qe["input_text"] = df_qe.apply(lambda r: f"EN: {r.get('src_en', '')}\nES_MT: {r.get('mt_es', '')}", axis=1)

hf_ds = Dataset.from_pandas(df_qe[["input_text", "hter_label"]])
splits = hf_ds.train_test_split(test_size=0.2, seed=42)
train_ds = splits["train"]
eval_ds  = splits["test"]
print("Train / Eval sizes:", len(train_ds), "/", len(eval_ds))


KeyError: "['hter_label'] not in index"

In [20]:
import jiwer

# Ensure required columns exist
required_cols = ["src_en", "mt_es", "pe_es"]
for c in required_cols:
    if c not in df_qe.columns:
        raise KeyError(f"Required column missing: {c}")

# Compute HTER if not present
if "hter_label" not in df_qe.columns:
    print("hter_label not found. Computing now...")

    def sentence_ter(hyp, ref):
        # Approximate HTER using word-level WER
        return jiwer.wer(str(ref), str(hyp))

    df_qe["hter_label"] = df_qe.apply(
        lambda r: sentence_ter(r["mt_es"], r["pe_es"]), axis=1
    )

    print("Computed hter_label. Sample:")
    display(df_qe[["src_en", "mt_es", "pe_es", "hter_label"]].head())
else:
    print("hter_label already exists.")


hter_label not found. Computing now...
Computed hter_label. Sample:


,src_en,mt_es,pe_es,hter_label
0,app store [TAG] apps,tienda de aplicaciones[TAG],Tienda de aplicaciones[TAG] aplicaciones,0.500000
1,tap 7 times on the background,toca 7 veces en el fondo de la pantalla,toca 7 veces el fondo de la pantalla,0.125000
2,pair,vincular,sincronizar,1.000000
3,Use [TAG] device to…,Usa [TAG] un dispositivo para…,Usa el dispositivo de [TAG] para…,0.666667
4,This will link [TAG] device with yours.,Esto vinculará [TAG] dispositivo con tuyo.,Esto vinculará el dispositivo de [TAG] con el ...,0.444444


In [21]:
df_qe["input_text"] = df_qe.apply(
    lambda r: f"EN: {r['src_en']}\nES_MT: {r['mt_es']}",
    axis=1
)

hf_ds = Dataset.from_pandas(df_qe[["input_text", "hter_label"]])
splits = hf_ds.train_test_split(test_size=0.2, seed=42)
train_ds = splits["train"]
eval_ds  = splits["test"]


In [22]:
MODEL_NAME = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    out = tokenizer(batch["input_text"], truncation=True, padding="max_length", max_length=256)
    out["labels"] = batch["hter_label"]
    return out

tokenized_train = train_ds.map(tokenize_fn, batched=True, remove_columns=["input_text","hter_label"])
tokenized_eval  = eval_ds.map(tokenize_fn, batched=True, remove_columns=["input_text","hter_label"])
tokenized_train.set_format(type="torch")
tokenized_eval.set_format(type="torch")
print("Tokenization done.")


Map:   0%|          | 0/54 [00:00<?, ? examples/s]

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

Tokenization done.


In [23]:
import numpy as np
import pandas as pd

def pearsonr_manual(a, b):
    a = np.array(a); b = np.array(b)
    if a.size == 0: return 0.0
    if np.std(a)==0 or np.std(b)==0: return 0.0
    return float(np.corrcoef(a, b)[0,1])

def spearmanr_manual(a, b):
    a_rank = pd.Series(a).rank().to_numpy()
    b_rank = pd.Series(b).rank().to_numpy()
    if np.std(a_rank)==0 or np.std(b_rank)==0: return 0.0
    return float(np.corrcoef(a_rank, b_rank)[0,1])

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.array(preds).squeeze()
    labels = np.array(labels).squeeze()
    mse = float(np.mean((preds - labels)**2))
    pearson = pearsonr_manual(preds, labels)
    spearman = spearmanr_manual(preds, labels)
    return {"pearson": pearson, "spearman": spearman, "mse": mse}


In [25]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1, problem_type="regression")

training_args = TrainingArguments(
    output_dir="./qe_mbert_regression",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    num_train_epochs=3,       # small: increase if you have GPU/time
    logging_steps=10,
    save_total_limit=1,
    # use small max_steps if you want very short run: add max_steps=50
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Start training (interrupt if you need faster demo)
trainer.train()
trainer.save_model("./qe_mbert_regression")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\santh\AppData\Local\Temp\ipykernel_9108\3004952667.py:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
C:\Users\santh\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,0.304400
20,0.420000
30,0.218000
40,0.386700


In [26]:
metrics = trainer.evaluate()
print("Evaluation metrics:", metrics)

# Predictions on eval set
preds_output = trainer.predict(tokenized_eval)
preds = np.array(preds_output.predictions).squeeze()
labels = np.array(preds_output.label_ids).squeeze()

# Save predictions with original text
eval_df = pd.DataFrame({
    "input_text": eval_ds["input_text"],
    "hter_true": labels,
    "hter_pred": preds
})
eval_df.to_csv("qe_eval_predictions.csv", index=False)
print("Saved qe_eval_predictions.csv")
display(eval_df.head(10))


C:\Users\santh\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Evaluation metrics: {'eval_loss': 0.5438607335090637, 'eval_pearson': 0.0332440638516417, 'eval_spearman': -0.029356144644223214, 'eval_mse': 0.5438607931137085, 'eval_runtime': 10.262, 'eval_samples_per_second': 1.364, 'eval_steps_per_second': 0.39, 'epoch': 3.0}
Saved qe_eval_predictions.csv


,input_text,hter_true,hter_pred
0,EN: Stay connected\nES_MT: Mantente conectado,1.000000,0.630502
1,EN: Jump right in!\nES_MT: ¡Saltar hacia la de...,1.333333,0.461997
2,EN: This will link [TAG] device with yours.\nE...,0.444444,0.533531
3,EN: You've reached the [TAG] person limit\nES_...,0.333333,0.602955
4,EN: DesignOne-of-a-kindfoldable design\n\nES_M...,3.000000,0.567218
5,EN: \nOSAndroid™ 13 + 3 upgradesMem / Storage8...,0.333333,0.468873
6,EN: Change the lens\nES_MT: Cómo cambiar el lente,0.666667,0.571778
7,"EN: Please, connect to Wi-Fi and try to downlo...",0.333333,0.474805
8,EN: USB Secure Folder Mode\nES_MT: Modo de car...,1.000000,0.496615
9,EN: CamerasHigh-res cams forsharper photos\nES...,0.285714,0.486983


In [27]:
# Correlation numbers and MSE
from math import isnan
pear = pearsonr_manual(preds, labels)
spear = spearmanr_manual(preds, labels)
mse = float(np.mean((preds-labels)**2))
print(f"Pearson: {pear:.4f}, Spearman: {spear:.4f}, MSE: {mse:.6f}")

# Example high/low predictions
eval_df["abs_err"] = (eval_df["hter_pred"] - eval_df["hter_true"]).abs()
display(eval_df.sort_values("abs_err", ascending=False).head(10))


Pearson: 0.0332, Spearman: -0.0294, MSE: 0.543861


,input_text,hter_true,hter_pred,abs_err
4,EN: DesignOne-of-a-kindfoldable design\n\nES_M...,3.000000,0.567218,2.432782
1,EN: Jump right in!\nES_MT: ¡Saltar hacia la de...,1.333333,0.461997,0.871337
8,EN: USB Secure Folder Mode\nES_MT: Modo de car...,1.000000,0.496615,0.503385
13,EN: Quick settings\nES_MT: configuración rápida,1.000000,0.570374,0.429626
0,EN: Stay connected\nES_MT: Mantente conectado,1.000000,0.630502,0.369498
11,EN: Go online to start playing\nES_MT: Conécta...,0.333333,0.633155,0.299822
12,EN: Onboarding tutorials\nES_MT: Tutoriales de...,0.333333,0.603505,0.270172
3,EN: You've reached the [TAG] person limit\nES_...,0.333333,0.602955,0.269622
9,EN: CamerasHigh-res cams forsharper photos\nES...,0.285714,0.486983,0.201269
10,EN: Family Space is not supported on this devi...,0.375000,0.549572,0.174572
